# Fixed-duration reconstruction (`recon_eval_tms.py`)
Back-to-back windows of 5 × `bin_ms` from the first frame, the same for every source. `fixed` settings use fixed bin edges (bin *b* = [b·D, (b+1)·D) ms of the window); `EVREAL-style` settings stretch the bins between the window's first and last event.

- **vs real** (`agreement_*`, the main result): each method's reconstruction compared with the reconstruction from the **real recorded events** of the same scene. Same network, same window: the closer, the more realistic the simulated events.
- **vs GT** (`gt_*`, shown only with `SHOW_GT = True`): each reconstruction compared with the **camera frame**. The `real` row is the reference level; simulators were given these frames, so beating `real` here does not mean more realistic events.
- MSE and LPIPS: lower is better; SSIM: higher is better.

vs GT exists only for windows ending within `gt_tol_ms` of a camera frame. Comparison images are saved in the report folder under `<setting>/figures/` (study) and `figures/` (between-frames run).

In [ ]:
from pathlib import Path
import json, os, sys
import numpy as np
import pandas as pd
from IPython.display import display, HTML, Image as IPImage

REPO = Path.cwd().resolve()
while not (REPO / "benchmarks" / "recon_eval.py").is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "benchmarks"))
import recon_eval
import recon_figures

REAL, EVTWEEN, HIGHER = recon_eval.REAL, set(recon_eval.EVTWEEN), recon_eval.HIGHER_IS_BETTER
ORDER = {m: i for i, m in enumerate((REAL, *recon_eval.METHODS))}
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)


def latest(base, marker):
    """Newest run folder under base that contains marker (i.e. a finished run)."""
    runs = sorted(p for p in Path(base).glob("*") if (p / marker).is_file())
    return runs[-1] if runs else None


def resolve(run, base, marker):
    if run is None:
        return None
    if run == "latest":
        found = latest(base, marker)
        if found is None:
            print(f"No finished run under {base}")
        return found
    path = Path(run) if Path(run).is_absolute() else REPO / run
    if not (path / marker).is_file():
        raise FileNotFoundError(f"{path} has no {marker} (not a finished run?)")
    return path


def show_figure(path):
    if path is None:
        print("   (no saved frames for this one)")
        return
    print("   saved:", path.relative_to(REPO) if path.is_relative_to(REPO) else path)
    display(IPImage(filename=str(path)))

import matplotlib.pyplot as plt

# ---- settings -------------------------------------------------------------
DATASET = "eds"
SPLIT = "test"
TMS_RUN = "latest"         # recon_eval_tms.py report folder, or "latest"
RECON_RUN = "latest"       # recon_eval.py report folder (between frames), "latest", or None to skip
METRIC = "lpips"           # vs-real metric for the main table and chart: "lpips", "ssim" or "mse"
SHOW_GT = True             # False: only the comparison with real events (tables, tests and images)

tms_dir = resolve(TMS_RUN, REPO / "reports" / "recon_tms" / DATASET / SPLIT, "summary_all.csv")
recon_dir = resolve(RECON_RUN, REPO / "reports" / "recon" / DATASET / SPLIT, "summary.csv")
NO_STUDY = ("No finished recon_eval_tms.py run: set TMS_RUN to a run folder, wait for the run to finish, or summarise "
            f"a run in progress with: python benchmarks/recon_eval_tms.py {DATASET} --summarize-only <run folder>")
print("study:      ", tms_dir)
print("recon_eval: ", recon_dir)

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    config = json.loads((tms_dir / "config.json").read_text())
    summary = pd.read_csv(tms_dir / "summary_all.csv")
    configs = sorted(summary["config"].unique(), key=lambda c: (c.split("_")[-1] != "ms", float(c[3:].split("ms")[0])))
    print("methods:", config["methods"])
    print("settings:", configs, "| warm-up", config["warmup_s"], "s | GT tolerance", config["gt_tol_ms"], "ms")


def cell(row, col):
    if pd.isna(row.get(f"{col}_mean")):
        return ""
    std = row.get(f"{col}_std")
    return f"{row[f'{col}_mean']:.3f}" + (f" ± {std:.3f}" if pd.notna(std) else "") + f" ({int(row[f'{col}_rank'])})"


def by_setting(col, include_real=False):
    s = summary if include_real else summary[summary["method"] != REAL]
    t = s.assign(value=s.apply(lambda r: cell(r, col), axis=1)).pivot_table(
        index=["network", "method"], columns="config", values="value", aggfunc="first")[configs].reset_index()
    t["_o"] = t["method"].map(ORDER)
    return t.sort_values(["network", "_o"]).drop(columns="_o").set_index(["network", "method"])

## Main result: vs real
Mean ± std over sequences, rank within each setting and network in brackets.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    print(f"agreement_{METRIC} ({'higher' if HIGHER[METRIC] else 'lower'} is better)")
    display(by_setting(f"agreement_{METRIC}"))

## How each method holds up as bins get shorter
One panel per network and bin mode; Evtween variants in colour, baselines gray and named at the line ends. Saved as a PNG in the report folder; the same numbers follow as a table.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
    GRAY, INK, MUTED = "#a3a29c", "#0b0b0b", "#52514e"
    col = f"agreement_{METRIC}_mean"
    data = summary[summary["method"] != REAL]
    nets = [n for n in recon_eval.NETWORKS if n in set(data["network"])]
    modes = [m for m in ("ms", "events") if m in set(data["bin_time"])]
    ours = [m for m in recon_eval.METHODS if m in EVTWEEN and m in set(data["method"])]
    color = {m: SLOTS[i] for i, m in enumerate(ours[:len(SLOTS)])}
    fig, axes = plt.subplots(len(nets), len(modes), figsize=(5.2 * len(modes), 3.6 * len(nets)),
                             sharex=True, sharey=True, squeeze=False)
    for i, net in enumerate(nets):
        for j, mode in enumerate(modes):
            ax = axes[i][j]
            d = data[(data["network"] == net) & (data["bin_time"] == mode)]
            ends = []
            for method, g in sorted(d.groupby("method"), key=lambda kv: (kv[0] in color, ORDER.get(kv[0], 99))):
                g = g.sort_values("bin_ms")
                mine = method in color
                ax.plot(g["bin_ms"], g[col], color=color.get(method, GRAY), lw=2 if mine else 1.2, marker="o",
                        ms=5 if mine else 3.5, zorder=3 if mine else 2, label=method if mine else None)
                ends.append([g[col].iloc[-1], g["bin_ms"].iloc[-1], method, INK if mine else MUTED])
            lo, hi = ax.get_ylim()
            gap = 0.045 * (hi - lo)
            placed = []
            for y, x, method, ink in sorted(ends, key=lambda e: e[0]):
                y_lab = max(y, placed[-1] + gap) if placed else y
                placed.append(y_lab)
                ax.annotate(method, (x, y), xytext=(x * 1.08, y_lab), textcoords="data", va="center", fontsize=7.5,
                            color=ink, annotation_clip=False)
            ax.set_ylim(lo, max(hi, placed[-1] + gap) if placed else hi)
            ax.set_xscale("log")
            ticks = sorted(d["bin_ms"].unique())
            ax.set_xticks(ticks); ax.set_xticklabels([f"{t:g}" for t in ticks]); ax.minorticks_off()
            ax.set_title(f"{net} · {'fixed-ms bins' if mode == 'ms' else 'EVREAL-style bins'}", fontsize=10, color=INK)
            ax.grid(axis="y", color="#e6e5e0", lw=0.8); ax.set_axisbelow(True)
            for side in ("top", "right"):
                ax.spines[side].set_visible(False)
            for side in ("left", "bottom"):
                ax.spines[side].set_color("#c9c8c2")
            ax.tick_params(colors=MUTED, labelsize=8)
            if j == 0:
                ax.set_ylabel(f"vs real {METRIC.upper()} ({'higher' if HIGHER[METRIC] else 'lower'} = closer to real)",
                              fontsize=8.5, color=MUTED)
            if i == len(nets) - 1:
                ax.set_xlabel("bin length (ms); window = 5 bins", fontsize=8.5, color=MUTED)
            ax.margins(x=0.25)
    handles, labels = axes[0][0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(4, len(handles)), frameon=False, fontsize=8.5)
    fig.tight_layout(rect=(0, 0, 1, 0.93 if handles else 1))
    out = tms_dir / f"agreement_{METRIC}_vs_bin_length.png"
    fig.savefig(out, dpi=150)
    plt.show()
    print("saved", out)
    display(pd.read_csv(tms_dir / "comparison.csv").set_index(["network", "bin_time", "method"]))

## vs GT (camera frames)
Only with `SHOW_GT = True`. Only windows ending within `gt_tol_ms` of a frame are scored; `gt_windows` says how many.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
elif not SHOW_GT:
    print("SHOW_GT = False: skipped")
else:
    display(summary.pivot_table(index=["network", "method"], columns="config", values="gt_windows",
                                aggfunc="first")[configs].rename(columns=lambda c: f"{c} (GT windows)").head(3))
    for m in ("lpips", "ssim", "mse"):
        print(f"gt_{m}")
        display(by_setting(f"gt_{m}", include_real=True))

## Sanity check of the inputs
Share of empty windows per source and setting (short bins can starve a source of events).

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    sanity = pd.concat([pd.read_csv(tms_dir / c / "sanity.csv") for c in configs if (tms_dir / c / "sanity.csv").is_file()])
    ok = sanity[sanity["status"] == "ok"]
    t = ok.groupby(["config", "method"]).agg(windows=("windows", "sum"), empty=("empty_windows", "sum")).reset_index()
    t["empty %"] = (100 * t["empty"] / t["windows"]).round(1)
    display(t.pivot_table(index="method", columns="config", values="empty %")[configs].sort_index(key=lambda s: s.map(ORDER)))
    display(sanity[sanity["status"] != "ok"])

## Best Evtween variant vs best baseline
Paired Wilcoxon test over sequences, per setting, network and metric.

In [ ]:
if tms_dir is None:
    print(NO_STUDY)
else:
    wx = pd.read_csv(tms_dir / "wilcoxon_all.csv")
    display(wx if SHOW_GT else wx[wx["metric"].str.startswith("agreement")])

## Comparison images of the saved windows
40 windows per sequence and setting (best/worst agreement, most real events, evenly spread). Saved to `<setting>/figures/` in the report folder.

In [ ]:
SHEET_CONFIG = None            # None = first setting
SHEET_SEQUENCE = None          # None = first sequence
SHEET_CATEGORY = "worst_agreement"   # "best_agreement" | "worst_agreement" | "high_activity" | "temporal"
SHEET_LIMIT = 3

if tms_dir is None:
    print(NO_STUDY)
elif not (tms_dir / "selection.csv").is_file():
    print("No saved windows (run with --save hybrid)")
else:
    selection = pd.read_csv(tms_dir / "selection.csv")
    display(selection.groupby(["config", "category"]).size().unstack(fill_value=0))
    cfg = SHEET_CONFIG or configs[0]
    sel = selection[selection["config"] == cfg]
    seq = SHEET_SEQUENCE or sel["sequence"].iloc[0]
    for _, row in sel[(sel["sequence"] == seq) & (sel["category"] == SHEET_CATEGORY)].head(SHEET_LIMIT).iterrows():
        print(f"{cfg} / {seq} / window {row.window} ({row.category})")
        show_figure(recon_figures.window_figure(tms_dir, cfg, seq, row.window, f"{row.category}_{seq}_w{int(row.window):06d}",
                                                show_gt=SHOW_GT, title=f"({row.category})"))

## Best results for one method
Ranks the saved frames (between frames) and windows (fixed duration) by one method's score:
- `FOCUS_MODE = "best"`: the method's own best scores;
- `FOCUS_MODE = "vs_baselines"`: where it beats the best baseline on the same frame by the widest margin.

Only saved frames can be shown, so this is the best among those. Images are saved to the report folders.

In [ ]:
FOCUS_METHOD = "evtween_v8_student"
FOCUS_METRIC = "agreement_lpips"   # agreement_* = vs real; gt_* = vs GT (needs SHOW_GT)
FOCUS_NETWORK = None               # "e2vid", "firenet", or None for the mean of both
FOCUS_MODE = "best"                # "best" | "vs_baselines"
FOCUS_TOP = 3
FOCUS_CONFIG = None                # study setting, e.g. "bin10ms_ms"; None = first


def rank_saved(scores, key):
    higher = HIGHER[FOCUS_METRIC.split("_", 1)[1]]
    s = scores[scores[FOCUS_METRIC].notna()]
    if FOCUS_NETWORK:
        s = s[s["network"] == FOCUS_NETWORK]
    s = s.groupby(["sequence", key, "method"])[FOCUS_METRIC].mean().unstack("method")
    if FOCUS_METHOD not in s:
        print(f"{FOCUS_METHOD} not in this run; methods: {sorted(s.columns)}")
        return None
    others = s.drop(columns=[REAL], errors="ignore")
    s["rank among methods"] = others.rank(axis=1, ascending=not higher, method="min")[FOCUS_METHOD]
    if FOCUS_MODE == "vs_baselines":
        base = [m for m in others.columns if m not in EVTWEEN]
        best_base = s[base].max(axis=1) if higher else s[base].min(axis=1)
        s["margin"] = (s[FOCUS_METHOD] - best_base) * (1 if higher else -1)
        s["best baseline"] = s[base].idxmax(axis=1) if higher else s[base].idxmin(axis=1)
        return s.sort_values("margin", ascending=False)[[FOCUS_METHOD, "best baseline", "margin", "rank among methods"]]
    return s.sort_values(FOCUS_METHOD, ascending=not higher)[[FOCUS_METHOD, "rank among methods"]]


if FOCUS_METRIC.startswith("gt_") and not SHOW_GT:
    print("FOCUS_METRIC is a GT metric but SHOW_GT = False; using agreement_lpips")
    FOCUS_METRIC = "agreement_lpips"
print(f"{FOCUS_METHOD}: {FOCUS_MODE} by {FOCUS_METRIC} ({'higher' if HIGHER[FOCUS_METRIC.split('_', 1)[1]] else 'lower'} is better)")
tag = f"{FOCUS_MODE}_{FOCUS_METHOD}_{FOCUS_METRIC}"
print("\n--- between frames (recon_eval.py) ---")
if recon_dir is not None and (recon_dir / "selection.csv").is_file():
    sel = pd.read_csv(recon_dir / "selection.csv")
    pf = pd.read_csv(recon_dir / "per_frame.csv").merge(sel[["sequence", "frame"]], on=["sequence", "frame"])
    top = rank_saved(pf, "frame")
    if top is not None:
        display(top.head(FOCUS_TOP).round(4))
        for i, ((seq, k), row) in enumerate(top.head(FOCUS_TOP).iterrows(), 1):
            show_figure(recon_figures.frame_figure(recon_dir, seq, k, f"{tag}_{i}_{seq}_{int(k):06d}", show_gt=SHOW_GT,
                                                   title=f"{FOCUS_METHOD} {FOCUS_MODE} #{i}"))
elif recon_dir is not None:
    print("No saved frames; make them from the stored metrics with\n"
          f"  python benchmarks/recon_eval.py {DATASET} --devices cuda:0 cuda:0 --run-dir {recon_dir} --save hybrid")
else:
    print("No recon_eval.py run selected")
print("\n--- fixed duration (recon_eval_tms.py) ---")
if tms_dir is None:
    print(NO_STUDY)
elif not (tms_dir / "selection.csv").is_file():
    print("No saved windows in the study run (run it with --save hybrid)")
else:
    cfg = FOCUS_CONFIG or configs[0]
    sel = pd.read_csv(tms_dir / "selection.csv")
    sel = sel[sel["config"] == cfg]
    pw = pd.read_csv(tms_dir / cfg / "per_window.csv").merge(sel[["sequence", "window"]], on=["sequence", "window"])
    print("setting:", cfg)
    top = rank_saved(pw, "window")
    if top is not None:
        display(top.head(FOCUS_TOP).round(4))
        for i, ((seq, w), row) in enumerate(top.head(FOCUS_TOP).iterrows(), 1):
            show_figure(recon_figures.window_figure(tms_dir, cfg, seq, w, f"{tag}_{i}_{seq}_w{int(w):06d}",
                                                    show_gt=SHOW_GT, title=f"{FOCUS_METHOD} {FOCUS_MODE} #{i}"))